# Notebook 1 — Data Cleaning & Gap Computation
**Challenge 5.2 — Community Behaviour & Participation Analysis on Welfare Schemes**  
IIT Roorkee Finance Club Open Projects 2026

## Objective
Load raw PM-KISAN district data, clean it, compute uptake gaps, and save analysis-ready CSV.

**Data sources used:**
- PM-KISAN district beneficiary data (pmkisan.gov.in / data.gov.in)
- Census 2011 agricultural household counts
- NFHS-5 district-level covariates (literacy, SC/ST share)
- MoSPI internet access proxy

> **Note:** This notebook uses a synthetic dataset calibrated to real India statistics (Census 2011 + NFHS-5) for reproducibility. Replace `pmkisan_district_raw.csv` with the actual downloaded file to run on real data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load raw data
raw = pd.read_csv('../data/raw/pmkisan_district_raw.csv')
print(f"Shape: {raw.shape}")
print("\nColumn types:")
print(raw.dtypes)
print("\nFirst 5 rows:")
raw.head()


In [ ]:
# Check missing values
print("Missing values per column:")
print(raw.isnull().sum())
print(f"\nTotal records: {len(raw)}")
print(f"States covered: {raw['state'].nunique()}")
print(f"\nState distribution:")
print(raw['state'].value_counts())


In [ ]:
# Standardise district names
raw['district_key'] = raw['district'].str.lower().str.strip()

# Compute uptake rate and gap
raw['uptake_rate'] = raw['pmkisan_enrolled'] / raw['agri_households']
raw['gap'] = raw['agri_households'] - raw['pmkisan_enrolled']
raw['gap_rate'] = (raw['gap'] / raw['agri_households']).clip(0, 1)
raw['uptake_rate_clipped'] = raw['uptake_rate'].clip(0, 1.05)

# Data quality flags
raw['data_quality_flag'] = 'ok'
raw.loc[raw['uptake_rate'] > 1.0, 'data_quality_flag'] = 'over-enrolled'
raw.loc[raw['agri_households'] < 10000, 'data_quality_flag'] = 'small-district'

print("Data quality flags:")
print(raw['data_quality_flag'].value_counts())


In [ ]:
# Keep only clean records
clean = raw[raw['data_quality_flag'] == 'ok'].copy()
print(f"Clean records: {len(clean)} / {len(raw)}")

# Summary statistics
print(f"\n--- Key Statistics ---")
print(f"National average uptake rate: {clean['uptake_rate'].mean():.1%}")
print(f"Median uptake rate:           {clean['uptake_rate'].median():.1%}")
print(f"Districts below 30% uptake:  {(clean['uptake_rate'] < 0.30).sum()}")
print(f"Districts below 50% uptake:  {(clean['uptake_rate'] < 0.50).sum()}")
print(f"Total estimated gap (households): {clean['gap'].sum():,.0f}")

# Save
clean.to_csv('../data/clean/pmkisan_clean.csv', index=False)
print("\nSaved: ../data/clean/pmkisan_clean.csv")


In [ ]:
# Visualise uptake distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(clean['uptake_rate'] * 100, bins=25, color='#4a90c4', edgecolor='white', alpha=0.85)
ax.axvline(clean['uptake_rate'].mean()*100, color='#e05a2b', linestyle='--', linewidth=2,
           label=f"Mean: {clean['uptake_rate'].mean():.0%}")
ax.set_xlabel('Uptake rate (%)'); ax.set_ylabel('Districts')
ax.set_title('PM-KISAN uptake rate distribution across 302 districts')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print("Notebook 1 complete ✓")
